# f6_m03b_errores_fpfn.ipynb
**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 6 — Interpretabilidad y Evaluación Final |
| **Módulo** | M03b — Análisis de Errores FP/FN |

---

## 🎯 Qué hace

Analiza el perfil de los errores del modelo: falsos positivos (FP) y falsos negativos (FN).
FP: alumno que NO abandona pero el modelo predice que sí — alarma falsa.
FN: alumno que SÍ abandona pero el modelo no lo detecta — el error más grave.
Analiza tanto variables numéricas (medias z-score) como categóricas
(rama y titulación más frecuentes en cada tipo de error).
Modelo usado: CatBoost__balanced con umbral estándar 0.5.

## 📋 Requisitos

- `data/05_modelado/X_test_prep.parquet`
- `data/05_modelado/X_test.parquet`
- `data/05_modelado/y_test.parquet`
- `data/05_modelado/models/CatBoost__balanced.pkl`
- `data/03_features/df_exp_target_eda.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `results/fase6/errores_fpfn.parquet` | Observaciones FP y FN con sus features |
| `results/fase6/errores_matriz_confusion.png` | Matriz de confusión |
| `results/fase6/errores_distribucion.png` | Distribución de probabilidades por tipo |
| `results/fase6/errores_perfil_features.png` | Perfil features numéricas FP vs FN vs correctos |
| `results/fase6/errores_perfil_categoricas.png` | Distribución rama FP vs FN |
| `docs/html/fase6/m03b_errores_fpfn.html` | Informe HTML |

## 🔄 Flujo

```
X_test_prep + X_test + CatBoost + df_exp_target_eda
    ↓ Calcular predicciones (umbral 0.5)
    ↓ Clasificar en TP/TN/FP/FN
    ↓ Definir features_num (X_test_prep cols) y features_cat (rama, titulacion)
    ↓ Matriz confusión → distribución probs → perfil numérico → perfil categórico
    → errores_fpfn.parquet + HTML
```

## ➡️ Siguiente

`f6_m04_robustez_indice.ipynb`


In [1]:
# ============================================================
# CELDA 1: CONFIGURACIÓN DE RUTAS
# ============================================================
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError('No se encontró src/ subiendo desde ' + str(start))

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

DIR_DATA     = ROOT / 'data' / '05_modelado'
DIR_FEATURES = ROOT / 'data' / '03_features'
DIR_MODELS   = ROOT / 'data' / '05_modelado' / 'models'
DIR_RESULTS  = ROOT / 'results' / 'fase6'
DIR_HTML     = ROOT / 'docs' / 'html' / 'fase6'
DIR_RESULTS.mkdir(parents=True, exist_ok=True)
DIR_HTML.mkdir(parents=True, exist_ok=True)

print(f'ROOT:        {ROOT}')
print(f'DIR_RESULTS: {DIR_RESULTS}')

ROOT:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
DIR_RESULTS: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\results\fase6


In [2]:
# ============================================================
# CELDA 2: IMPORTS
# ============================================================
import base64
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import confusion_matrix, classification_report
from src.html.render import render_pagina_desde_fichero

matplotlib.rcParams['figure.dpi'] = 120
print('Imports OK.')
from src.config_entorno import RAMAS_NOMBRES, NOMBRES_LEGIBLES_FEATURES

def nombre_legible(f): return NOMBRES_LEGIBLES_FEATURES.get(f, f.replace("_"," "))


Imports OK.


In [3]:
# ============================================================
# CELDA 3: CARGAR DATOS Y MODELO
# ============================================================
X_test       = pd.read_parquet(DIR_DATA / 'X_test.parquet')
X_test_prep  = pd.read_parquet(DIR_DATA / 'X_test_prep.parquet')
y_test       = pd.read_parquet(DIR_DATA / 'y_test.parquet').squeeze()
pipeline_cat = joblib.load(DIR_MODELS / 'CatBoost__balanced.pkl')

y_true = y_test.values.ravel()
y_prob = pipeline_cat.predict_proba(X_test_prep)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print(f'X_test_prep: {X_test_prep.shape}')
print(f'Abandono real:     {y_true.mean():.2%} ({y_true.sum()} alumnos)')
print(f'Abandono predicho: {y_pred.mean():.2%} ({y_pred.sum()} alumnos)')
print()
print(classification_report(y_true, y_pred, target_names=['No abandona', 'Abandona']))

X_test_prep: (6725, 27)
Abandono real:     29.25% (1967 alumnos)
Abandono predicho: 33.19% (2232 alumnos)

              precision    recall  f1-score   support

 No abandona       0.94      0.89      0.92      4758
    Abandona       0.77      0.87      0.82      1967

    accuracy                           0.89      6725
   macro avg       0.86      0.88      0.87      6725
weighted avg       0.89      0.89      0.89      6725



In [4]:
# ============================================================
# CELDA 4: CLASIFICAR EN TP/TN/FP/FN + DEFINIR TIPOS DE FEATURES
# Se definen aquí una vez para usarlas en todas las celdas:
#   features_num: columnas numéricas de X_test_prep
#   features_cat: variables de contexto recuperadas por join
# Esto evita el TypeError al llamar .mean() sobre strings.
# ============================================================
tipo = np.where(
    (y_true == 1) & (y_pred == 1), 'TP',
    np.where(
        (y_true == 0) & (y_pred == 0), 'TN',
        np.where((y_true == 0) & (y_pred == 1), 'FP', 'FN')
    )
)

# Features numéricas del modelo — excluir rama aunque esté label-encoded
# rama es semánticamente categórica aunque X_test_prep la tenga como int
FEATURES_CATEGORICAS_REALES = ['rama', 'sexo', 'via_acceso', 'situacion_laboral',
                                'pais_nombre', 'provincia', 'universidad_origen']
features_num = [c for c in X_test_prep.columns
                if pd.api.types.is_numeric_dtype(X_test_prep[c])
                and c not in FEATURES_CATEGORICAS_REALES]

# Features categóricas de contexto (recuperadas por join)
features_cat = ['titulacion', 'rama']

# Recuperación de contexto: titulacion y rama
df_ref  = pd.read_parquet(DIR_FEATURES / 'df_exp_target_eda.parquet')
df_meta = df_ref[['titulacion', 'rama']].iloc[X_test.index].copy()
df_meta.index = X_test.index

# Construir df_errores con numéricas + categóricas + metadatos clasificación
# Orden: primero las features del modelo, luego el contexto añadido
df_errores = X_test_prep.copy()           # 27 features numéricas del modelo
df_errores['titulacion'] = df_meta['titulacion'].values   # contexto (str)
df_errores['rama']       = df_meta['rama'].values          # contexto (str)
df_errores['tipo']       = tipo            # metadato clasificación (str)
df_errores['y_true']     = y_true          # etiqueta real (int)
df_errores['y_pred']     = y_pred          # predicción (int)
df_errores['y_prob']     = y_prob          # probabilidad (float)

conteo = df_errores['tipo'].value_counts()
total  = len(df_errores)
for t in ['TP', 'TN', 'FP', 'FN']:
    n = conteo.get(t, 0)
    print(f'{t}: {n:5d} ({n/total:.1%})')

print(f'\nFeatures numéricas del modelo: {len(features_num)}')
print(f'Features categóricas contexto: {features_cat}')

df_errores.to_parquet(DIR_RESULTS / 'errores_fpfn.parquet')
print('\n✅ errores_fpfn.parquet guardado.')

TP:  1719 (25.6%)
TN:  4245 (63.1%)
FP:   513 (7.6%)
FN:   248 (3.7%)

Features numéricas del modelo: 20
Features categóricas contexto: ['titulacion', 'rama']

✅ errores_fpfn.parquet guardado.


In [5]:
# ============================================================
# CELDA 5: GRÁFICO 1 — MATRIZ DE CONFUSIÓN
# ============================================================
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicho:\nNo abandona', 'Predicho:\nAbandona'], fontsize=11)
ax.set_yticklabels(['Real:\nNo abandona', 'Real:\nAbandona'], fontsize=11)

tipos_cuadrante = [['TN', 'FP'], ['FN', 'TP']]
colores_texto   = [['#2d3748', '#e53e3e'], ['#e53e3e', '#2d3748']]
for i in range(2):
    for j in range(2):
        pct = cm[i, j] / total
        ax.text(j, i, f'{tipos_cuadrante[i][j]}\n{cm[i,j]:,}\n({pct:.1%})',
                ha='center', va='center', fontsize=12, fontweight='bold',
                color=colores_texto[i][j])

ax.set_title('Fase 6 — Matriz de confusión (CatBoost, umbral 0.5)', fontsize=13, pad=12)
plt.colorbar(im, ax=ax)
plt.tight_layout()
ruta_cm = DIR_RESULTS / 'errores_matriz_confusion.png'
plt.savefig(ruta_cm, dpi=120, bbox_inches='tight')
plt.close()
print('✅ Matriz de confusión guardada.')

✅ Matriz de confusión guardada.


In [6]:
# ============================================================
# CELDA 6: GRÁFICO 2 — DISTRIBUCIÓN DE PROBABILIDADES POR TIPO
# FN con prob baja = errores difíciles (sin señal).
# FN con prob media = evitables bajando el umbral.
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)
colores_tipo = {'TP': '#38a169', 'TN': '#3182ce', 'FP': '#dd6b20', 'FN': '#e53e3e'}

for ax, tipo_plot in zip(axes, ['TP', 'TN', 'FP', 'FN']):
    datos = df_errores[df_errores['tipo'] == tipo_plot]['y_prob']
    ax.hist(datos, bins=20, color=colores_tipo[tipo_plot], alpha=0.8, edgecolor='white')
    ax.set_title(f'{tipo_plot} (n={len(datos)})', fontsize=11)
    ax.set_xlabel('Probabilidad predicha')
    ax.axvline(0.5, color='gray', linestyle='--', linewidth=1)
    ax.set_xlim(0, 1)

plt.suptitle('Fase 6 — Distribución de probabilidades por tipo', fontsize=13, y=1.02)
plt.tight_layout()
ruta_dist = DIR_RESULTS / 'errores_distribucion.png'
plt.savefig(ruta_dist, dpi=120, bbox_inches='tight')
plt.close()
print('✅ Distribución guardada.')

✅ Distribución guardada.


In [7]:
# ============================================================
# CELDA 7: GRÁFICO 3 — PERFIL NUMÉRICO: FP vs FN vs CORRECTOS
# Usamos features_num (definidas en celda 4) — solo las 27
# features del modelo, sin las columnas de contexto añadidas.
# Z-score respecto a la media global para comparar escalas.
# ============================================================
mask_ok = np.isin(tipo, ['TP', 'TN'])
mask_fp = tipo == 'FP'
mask_fn = tipo == 'FN'

medias_global = X_test_prep[features_num].mean()
stds_global   = X_test_prep[features_num].std().replace(0, 1)

df_perfil = pd.DataFrame({
    'Correctos (TP+TN)': (X_test_prep[mask_ok][features_num].mean() - medias_global) / stds_global,
    'Falsos Positivos':  (X_test_prep[mask_fp][features_num].mean() - medias_global) / stds_global,
    'Falsos Negativos':  (X_test_prep[mask_fn][features_num].mean() - medias_global) / stds_global,
})

df_perfil['diff_fp_fn'] = (df_perfil['Falsos Positivos'] - df_perfil['Falsos Negativos']).abs()
top_feats = df_perfil.nlargest(10, 'diff_fp_fn').index.tolist()
df_plot   = df_perfil.loc[top_feats, ['Correctos (TP+TN)', 'Falsos Positivos', 'Falsos Negativos']]

fig, ax = plt.subplots(figsize=(12, 6))
x     = np.arange(len(top_feats))
ancho = 0.25
colores_grupos = ['#3182ce', '#dd6b20', '#e53e3e']

for i, (col, color) in enumerate(zip(df_plot.columns, colores_grupos)):
    ax.bar(x + i * ancho, df_plot[col], ancho, label=col, color=color, alpha=0.8)

ax.set_xticks(x + ancho)
ax.set_xticklabels([nombre_legible(f) for f in top_feats], rotation=35, ha='right', fontsize=9)
ax.axhline(0, color='gray', linewidth=0.8)
ax.set_ylabel('Desviación respecto a la media global (z-score)')
ax.set_title('Fase 6 — Perfil numérico: Correctos vs FP vs FN (top 10 diferencias)', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
ruta_perfil = DIR_RESULTS / 'errores_perfil_features.png'
plt.savefig(ruta_perfil, dpi=120, bbox_inches='tight')
plt.close()
print('✅ Perfil numérico guardado.')

✅ Perfil numérico guardado.


In [8]:
# ============================================================
# CELDA 8: GRÁFICO 4 — PERFIL CATEGÓRICO: RAMA EN FP vs FN
# Usamos features_cat (definidas en celda 4): titulacion y rama.
# Para categóricas no se puede calcular media — se usa
# distribución relativa (% de cada grupo en FP vs FN).
# ============================================================
fn = df_errores[df_errores['tipo'] == 'FN']
fp = df_errores[df_errores['tipo'] == 'FP']

# Distribución de rama en FP vs FN (normalizada)
rama_fn = fn['rama'].value_counts(normalize=True)
rama_fp = fp['rama'].value_counts(normalize=True)
ramas_abr = sorted(set(rama_fn.index) | set(rama_fp.index))
ramas     = [RAMAS_NOMBRES.get(r, r) for r in ramas_abr]

fig, ax = plt.subplots(figsize=(10, 5))
x     = np.arange(len(ramas))
ancho = 0.35
ax.bar(x - ancho/2, [rama_fn.get(r, 0) for r in ramas_abr], ancho,
       label='Falsos Negativos', color='#e53e3e', alpha=0.8)
ax.bar(x + ancho/2, [rama_fp.get(r, 0) for r in ramas_abr], ancho,
       label='Falsos Positivos', color='#dd6b20', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(ramas, fontsize=11)
ax.set_ylabel('Proporción dentro del grupo de error')
ax.set_title('Fase 6 — Distribución de rama en FP vs FN', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
ruta_cat = DIR_RESULTS / 'errores_perfil_categoricas.png'
plt.savefig(ruta_cat, dpi=120, bbox_inches='tight')
plt.close()
print('✅ Perfil categórico guardado.')

✅ Perfil categórico guardado.


In [9]:
# ============================================================
# CELDA 9: ANÁLISIS TEXTUAL — PATRONES EN ERRORES
# Numérico: media de features_num por grupo de error.
# Categórico: distribución de rama y top titulaciones.
# ============================================================
print('=== FALSOS NEGATIVOS (abandonan pero no detectados) ===')
print(f'Total FN: {len(fn)} ({len(fn)/total:.1%} del test)')
print(f'Prob media predicha: {fn["y_prob"].mean():.3f}')
print('\nDistribución por rama:')
print(fn['rama'].value_counts().to_string())
print('\nTop 5 titulaciones con más FN:')
print(fn['titulacion'].value_counts().head(5).to_string())
print('\nMedia features numéricas (top 10 por valor absoluto):')
medias_fn = fn[features_num].mean()
print(medias_fn.abs().nlargest(10).round(3).to_string())

print('\n=== FALSOS POSITIVOS (no abandonan pero modelo alerta) ===')
print(f'Total FP: {len(fp)} ({len(fp)/total:.1%} del test)')
print(f'Prob media predicha: {fp["y_prob"].mean():.3f}')
print('\nDistribución por rama:')
print(fp['rama'].value_counts().to_string())
print('\nTop 5 titulaciones con más FP:')
print(fp['titulacion'].value_counts().head(5).to_string())
print('\nMedia features numéricas (top 10 por valor absoluto):')
medias_fp = fp[features_num].mean()
print(medias_fp.abs().nlargest(10).round(3).to_string())

print('\n=== COMPARATIVA FN vs FP — top 5 diferencias numéricas ===')
diff = (medias_fn - medias_fp).abs().nlargest(5)
print(diff.round(3).to_string())

=== FALSOS NEGATIVOS (abandonan pero no detectados) ===
Total FN: 248 (3.7% del test)
Prob media predicha: 0.274

Distribución por rama:
rama
SO    124
TE     73
HU     31
SA     12
EX      8

Top 5 titulaciones con más FN:
titulacion
Grado en Derecho                                  24
Grado en Finanzas y Contabilidad                  15
Grado en Comunicación Audiovisual                 14
Grado en Humanidades: Estudios Interculturales    14
Grado en Diseño y Desarrollo de Videojuegos       14

Media features numéricas (top 10 por valor absoluto):
tasa_abandono_titulacion    0.565
edad_entrada                0.323
tasa_repeticion             0.280
cred_repetidos              0.265
nota_1er_anio               0.204
n_anios_trabajando          0.181
max_pagos                   0.161
cupo                        0.153
cred_superados_anio_1er     0.153
orden_preferencia           0.119

=== FALSOS POSITIVOS (no abandonan pero modelo alerta) ===
Total FP: 513 (7.6% del test)
Prob media pred

In [10]:
# ============================================================
# CELDA 10: GENERAR HTML
# render_pagina_desde_fichero — estándar del proyecto.
# ============================================================
def img_b64(ruta: Path) -> str:
    if not ruta.exists():
        return ''
    with open(ruta, 'rb') as f:
        return base64.b64encode(f.read()).decode()

def bloque_imagen(b64: str, titulo: str, caption: str) -> str:
    if not b64:
        return f'<p style="color:#e53e3e">⚠️ Imagen no disponible: {titulo}</p>'
    return (
        '<div style="margin:24px 0">'
        f'<h3 style="color:#2d3748;font-size:15px">{titulo}</h3>'
        f'<img src="data:image/png;base64,{b64}" '
        'style="max-width:100%;border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,.1)">'
        f'<p style="color:#718096;font-size:12px;margin-top:6px">{caption}</p>'
        '</div>'
    )

n_tp = conteo.get('TP', 0)
n_tn = conteo.get('TN', 0)
n_fp = conteo.get('FP', 0)
n_fn = conteo.get('FN', 0)

# Top 3 titulaciones FN y FP para el HTML
top3_fn = fn['titulacion'].value_counts().head(3)
top3_fp = fp['titulacion'].value_counts().head(3)
filas_tit = ''
for tit in set(top3_fn.index) | set(top3_fp.index):
    n_fn_tit = top3_fn.get(tit, 0)
    n_fp_tit = top3_fp.get(tit, 0)
    filas_tit += (
        f'<tr><td style="padding:6px 12px;font-size:12px">{tit[:50]}</td>'
        f'<td style="padding:6px 12px;text-align:center;color:#e53e3e">{n_fn_tit}</td>'
        f'<td style="padding:6px 12px;text-align:center;color:#dd6b20">{n_fp_tit}</td></tr>'
    )

contenido = (
    '<h2 style="color:#2d3748">Fase 6 — Análisis de Errores: FP y FN</h2>'
    '<p style="color:#4a5568;font-size:14px;max-width:800px">'
    'Caracterización de los errores del modelo CatBoost con umbral de decisión 0.5. '
    'Se analiza tanto el perfil numérico (features del modelo) como el perfil '
    'categórico (rama y titulación) de los alumnos mal clasificados.'
    '</p>'
    '<div style="display:flex;gap:16px;margin:20px 0;flex-wrap:wrap">'
    f'<div style="padding:16px 24px;background:#f0fff4;border-radius:8px;text-align:center">'
    f'<div style="font-size:28px;font-weight:700;color:#38a169">{n_tp:,}</div>'
    f'<div style="font-size:12px;color:#718096">Verdaderos positivos<br>({n_tp/total:.1%})</div></div>'
    f'<div style="padding:16px 24px;background:#ebf8ff;border-radius:8px;text-align:center">'
    f'<div style="font-size:28px;font-weight:700;color:#3182ce">{n_tn:,}</div>'
    f'<div style="font-size:12px;color:#718096">Verdaderos negativos<br>({n_tn/total:.1%})</div></div>'
    f'<div style="padding:16px 24px;background:#fffbeb;border-radius:8px;text-align:center">'
    f'<div style="font-size:28px;font-weight:700;color:#dd6b20">{n_fp:,}</div>'
    f'<div style="font-size:12px;color:#718096">Falsos positivos<br>({n_fp/total:.1%})</div></div>'
    f'<div style="padding:16px 24px;background:#fff5f5;border-radius:8px;text-align:center">'
    f'<div style="font-size:28px;font-weight:700;color:#e53e3e">{n_fn:,}</div>'
    f'<div style="font-size:12px;color:#718096">Falsos negativos<br>({n_fn/total:.1%})</div></div>'
    '</div>'
    + bloque_imagen(img_b64(ruta_cm),
        'Matriz de confusión',
        'FN (rojo, abajo-izquierda): alumnos que abandonan sin ser detectados — mayor coste social.')
    + bloque_imagen(img_b64(ruta_dist),
        'Distribución de probabilidades por tipo de clasificación',
        'FN con prob baja = errores difíciles (sin señal). '
        'FN con prob media (~0.4-0.5) = evitables bajando el umbral.')
    + bloque_imagen(img_b64(ruta_perfil),
        'Perfil de features numéricas — Correctos vs FP vs FN',
        'Z-score respecto a la media global. Muestra qué variables del modelo '
        'distinguen a los alumnos mal clasificados de los correctamente clasificados.')
    + bloque_imagen(img_b64(ruta_cat),
        'Distribución de rama en FP vs FN',
        'Proporción de cada rama dentro del grupo de error. '
        'Diferencias entre FP y FN revelan si el modelo falla más en ramas concretas.')
    + '<h3 style="color:#2d3748;margin-top:24px">Titulaciones con más errores</h3>'
    + '<table style="width:100%;border-collapse:collapse;font-size:13px">'
    + '<thead><tr style="background:#edf2f7">'
    + '<th style="padding:6px 12px;text-align:left">Titulación</th>'
    + '<th style="padding:6px 12px;text-align:center;color:#e53e3e">FN</th>'
    + '<th style="padding:6px 12px;text-align:center;color:#dd6b20">FP</th>'
    + '</tr></thead>'
    + f'<tbody>{filas_tit}</tbody></table>'
    + '<div style="margin-top:24px;padding:16px;background:#fff5f5;'
    'border-left:4px solid #e53e3e;border-radius:6px;font-size:13px;color:#742a2a">'
    '<strong>Coste asimétrico:</strong> Un falso negativo tiene mayor coste social '
    'que un falso positivo. Si la institución prioriza no dejar pasar ningún caso, '
    'se recomienda bajar el umbral de decisión por debajo de 0.5.'
    '</div>'
)

html_completo = render_pagina_desde_fichero('f6_m03b_errores_fpfn.ipynb', contenido)
ruta_html = DIR_HTML / 'm03b_errores_fpfn.html'
ruta_html.write_text(html_completo, encoding='utf-8')
print(f'✅ HTML generado: {ruta_html}')

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase6\m03b_errores_fpfn.html
